In [1]:
from bs4 import BeautifulSoup
import requests
import cloudscraper
import time
import random
import re
import logging

In [2]:
team_hrefs = {
    "Arizona Cardinals": "crd",
    "Baltimore Colts": "clt",
    "St. Louis Cardinals": "crd",
    "Boston Patriots": "nwe",
    "Chicago Bears": "chi",
    "Green Bay Packers": "gnb",
    "New York Giants": "nyg",
    "Detroit Lions": "det",
    "Washington Commanders": "was",
    "Washington Football Team": "was",
    "Washington Redskins": "was",
    "Philadelphia Eagles": "phi",
    "Pittsburgh Steelers": "pit",
    "Los Angeles Chargers": "sdg",
    "San Francisco 49ers": "sfo",
    "Houston Oilers": "oti",
    "Cleveland Browns": "cle",
    "Indianapolis Colts": "clt",
    "Dallas Cowboys": "dal",
    "Kansas City Chiefs": "kan",
    "Los Angeles Rams": "ram",
    "Denver Broncos": "den",
    "New York Jets": "nyj",
    "New England Patriots": "nwe",
    "Las Vegas Raiders": "rai",
    "Tennessee Titans": "oti",
    "Tennessee Oilers": "oti",
    "Phoenix Cardinals": "crd",
    "Los Angeles Raiders": "rai",
    "Buffalo Bills": "buf",
    "Minnesota Vikings": "min",
    "Atlanta Falcons": "atl",
    "Miami Dolphins": "mia",
    "New Orleans Saints": "nor",
    "Cincinnati Bengals": "cin",
    "Seattle Seahawks": "sea",
    "Tampa Bay Buccaneers": "tam",
    "Carolina Panthers": "car",
    "Jacksonville Jaguars": "jax",
    "Baltimore Ravens": "rav",
    "Houston Texans": "htx",
    "Oakland Raiders": "rai",
    "San Diego Chargers": "sdg",
    "St. Louis Rams": "ram",
    "Boston Patriots": "nwe",
}

In [3]:
logging.basicConfig(
    filename="web_scraping.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

In [4]:
scraper = cloudscraper.create_scraper()

In [5]:
def get_number_of_weeks(season: int) -> int:
    """Return the number of completed weeks in a season."""
    weeks = 0
    while True:
        url = f"https://www.pro-football-reference.com/years/{season}/week_{weeks + 1}.htm"
        response = scraper.get(url)

        # Stop if the page does not exist (404 or other errors)
        if response.status_code != 200:
            break

        # Stop if the page is a preview for a future week
        title = response.text.split("<title>")[1].split("</title>")[0]
        if "Preview" in title:
            break

        weeks += 1
        time.sleep(random.uniform(1, 3))

    return weeks

In [6]:
def get_week_links(season: int, week: int) -> list[str]:
    """Get the links to all games in a single week."""
    url = f"https://www.pro-football-reference.com/years/{season}/week_{week}.htm"
    response = scraper.get(url)

    if response.status_code != 200:
        logging.warning(f"Failed to scrape week {week} of {season}. Status code: {response.status_code}")
        return []
    
    soup = BeautifulSoup(response.text, "lxml")
    return [
        f"https://www.pro-football-reference.com{cell.a['href']}"
        for cell in soup.find_all("td", class_="gamelink")
    ]

In [7]:
def get_season_links(season: int) -> list[str]:
    """Get the links to all games in a season."""
    season_links = []
    number_of_weeks = get_number_of_weeks(season)

    for week in range(1, number_of_weeks + 1):
        week_links = get_week_links(season, week)
        season_links.extend(week_links)
        time.sleep(random.uniform(1, 3))
        
    return season_links

In [7]:
test_links = get_season_links(2010)
len(test_links) #Should be 267 games

267

In [8]:
url = "https://www.pro-football-reference.com/boxscores/201009090nor.htm"

In [9]:
def get_soup(game_url: str) -> BeautifulSoup:
    """Return BeautifulSoup object for a given game URL."""
    response = scraper.get(game_url).text
    clean_response = re.sub(r"<!--|-->", "", response)
    soup = BeautifulSoup(clean_response, "lxml")
    return soup

In [10]:
def scrape_game_info(soup: BeautifulSoup) -> dict:
    """Scrape key game information for a single game.
    
    Args:
        soup: BeautifulSoup object of the game page.

    Returns:
        game_info: Dictionary containing home and away teams, game metadata,
        (roof, surface, weather, attendance), home line, and over/under.
    """
    # Initialize dictionary with all expected keys as None
    game_info = {
        "home_team": None,
        "away_team": None,
        "roof": None,
        "surface": None,
        "weather": None,
        "attendance": None,
        "home_line": None,
        "over_under": None
    }

    # Scrape home and away teams
    home_team = soup.find("table", id="scoring").find("th", {"data-stat": "home_team_score"}).text.lower()
    away_team = soup.find("table", id="scoring").find("th", {"data-stat": "vis_team_score"}).text.lower()
    game_info["home_team"] = home_team
    game_info["away_team"] = away_team

    # Scrape game metadata and betting lines
    table = soup.find("table", id="game_info")
    for row in table.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue
        field_name = th.text.strip().lower()

        if field_name == "roof":
            game_info["roof"] = td.text
        elif field_name == "surface":
            game_info["surface"] = td.text
        elif field_name == "weather":
            game_info["weather"] = td.text
        elif field_name == "attendance":
            game_info["attendance"] = int(td.text.replace(",", ""))
        elif field_name == "vegas line":
            vegas_favorite = " ".join(td.text.split()[0:-1])
            vegas_line = float(td.text.split()[-1])
            if team_hrefs[vegas_favorite] == home_team:
                game_info["home_line"] = vegas_line
            elif team_hrefs[vegas_favorite] == away_team:
                game_info["home_line"] = -vegas_line
            else:
                raise ValueError("Vegas favorite does not match either team.")
        elif field_name == "over/under":
            game_info["over_under"] = float(td.text.split()[0])

    return game_info

In [11]:
def scrape_expected_points_summary(soup: BeautifulSoup) -> dict:
    """Scrape expected points summary for a single game.
    
    Args:
        soup: BeautifulSoup object of the game page.

    Returns:
        expected_points: Dictionary containing expected points data for each team,
        keyed by team abbreviation.
    """
    table = soup.find("table", id="expected_points")
    expected_points = {}

    # Iterate over each row in the table
    for row in table.find_all("tr"):
        th = row.find("th", {"data-stat": "team_name"})
        if not th or th.get("scope") == "col":
            continue # Skip headers

        # Map team name to abbreviation
        team_name = th.text.strip()
        for key in team_hrefs.keys():
            if team_name in key:
                team_abbr = team_hrefs[key]
                break

        # Build the dictionary for this team
        expected_points[team_abbr] = {}
        for td in row.find_all("td"):
            expected_points[team_abbr][td["data-stat"]] = float(td.text)

    return expected_points


In [12]:
soup = get_soup(url)

In [13]:
scrape_game_info(soup)

{'home_team': 'nor',
 'away_team': 'min',
 'roof': 'dome',
 'surface': 'sportturf',
 'weather': None,
 'attendance': 70051,
 'home_line': -5.0,
 'over_under': 49.5}

In [14]:
scrape_expected_points_summary(soup)

{'min': {'pbp_exp_points_tot': -5.0,
  'pbp_exp_points_off_tot': -6.45,
  'pbp_exp_points_off_pass': -5.54,
  'pbp_exp_points_off_rush': -0.91,
  'pbp_exp_points_off_to': -4.17,
  'pbp_exp_points_def_tot': -2.86,
  'pbp_exp_points_def_pass': -4.6,
  'pbp_exp_points_def_rush': 1.42,
  'pbp_exp_points_def_to': 0.0,
  'pbp_exp_points_st': 4.34,
  'pbp_exp_points_k': -2.22,
  'pbp_exp_points_kr': 0.29,
  'pbp_exp_points_p': 4.47,
  'pbp_exp_points_pr': -4.13,
  'pbp_exp_points_fgxp': 5.93},
 'nor': {'pbp_exp_points_tot': 5.0,
  'pbp_exp_points_off_tot': 2.86,
  'pbp_exp_points_off_pass': 4.6,
  'pbp_exp_points_off_rush': -1.42,
  'pbp_exp_points_off_to': 0.0,
  'pbp_exp_points_def_tot': 6.45,
  'pbp_exp_points_def_pass': 5.54,
  'pbp_exp_points_def_rush': 0.91,
  'pbp_exp_points_def_to': 4.17,
  'pbp_exp_points_st': -4.34,
  'pbp_exp_points_k': -0.29,
  'pbp_exp_points_kr': 2.22,
  'pbp_exp_points_p': 4.13,
  'pbp_exp_points_pr': -4.47,
  'pbp_exp_points_fgxp': -5.93}}